In [1]:
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. 데이터 로드 (희소 행렬 및 로그 변환된 타겟)
X_full_clean = pd.read_pickle("../data/processed/X_full_clean.pkl")
y_log = pd.read_pickle("../data/processed/y_log.pkl")

# 2. 학습/테스트 데이터 분할 (동일한 random_state로 일관성 유지)
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_full_clean, y_log, test_size=0.2, random_state=42
)

train_data = lgb.Dataset(X_train_f, label=y_train_f, categorical_feature=['brand_name'])
valid_data = lgb.Dataset(X_test_f, label=y_test_f, reference=train_data, categorical_feature=['brand_name'])


# 3. RMSLE 평가 함수 정의
def rmsle(y_true_orig, y_pred_orig):
  y_pred_orig = np.clip(y_pred_orig, 0, None)
  return np.sqrt(np.mean((np.log1p(y_pred_orig) - np.log1p(y_true_orig)) ** 2))

In [ ]:
# # brand_name은 category 타입으로 변환 (LightGBM이 직접 처리)
# X_full['brand_name'] = X_full['brand_name'].astype('category')

# # 나머지 4개 텍스트 컬럼은 제거 (이미 TF-IDF로 별도 처리됨)
# drop_cols = ['name', 'category_name', 'item_description', 'text']
# X_full_clean = X_full.drop(columns=drop_cols)

# print(X_full_clean.dtypes.value_counts())
# print(X_full_clean.shape)  # (1481611, 1000) 나와야 함

float64     995
int64         4
category      1
Name: count, dtype: int64
(1481611, 1000)


In [ ]:
# X_full_clean.to_pickle("../data/processed/X_full_clean.pkl")

In [ ]:
# from sklearn.model_selection import train_test_split
# import numpy as np
# import lightgbm as lgb

# X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
#     X_full_clean, y_log, test_size=0.2, random_state=42
# )

# train_data = lgb.Dataset(X_train_f, label=y_train_f, categorical_feature=['brand_name'])
# valid_data = lgb.Dataset(X_test_f, label=y_test_f, reference=train_data, categorical_feature=['brand_name'])

In [ ]:
# import re

# def clean_col_name(col):
#     return re.sub(r'[^A-Za-z0-9_]+', '_', str(col))

# X_full_clean.columns = [clean_col_name(col) for col in X_full_clean.columns]

# print(X_full_clean.columns[:20].tolist())  # 확인

['item_condition_id', 'brand_name', 'price', 'shipping', 'name_length', 'description_length', 'log_price', 'category_1_Electronics', 'category_1_Handmade', 'category_1_Home', 'category_1_Kids', 'category_1_Men', 'category_1_Other', 'category_1_Sports_Outdoors', 'category_1_Unknown', 'category_1_Vintage_Collectibles', 'category_1_Women', 'category_2_Antique', 'category_2_Apparel', 'category_2_Art']


In [ ]:
# dup_cols = X_full_clean.columns[X_full_clean.columns.duplicated()]
# print("중복된 컬럼명:", dup_cols.tolist())

중복된 컬럼명: []


In [2]:
# # leak_cols = ['price', 'log_price']
# # X_full_clean = X_full_clean.drop(columns=leak_cols)

# print(X_full_clean.shape)
# print([col for col in X_full_clean.columns if 'price' in col.lower()])  # 빈 리스트여야 정상

(1481611, 997)
[]


In [ ]:
# # 'price'나 타겟과 관련된 단어가 포함된 컬럼들을 찾아내서 모두 제거
# cols_to_drop = [
#     col for col in X_full_clean.columns if 'price' in col.lower()
# ]
# # print("제거할 컬럼들:", cols_to_drop)

# # 실제 제거 수행
# # X_full_clean = X_full_clean.drop(columns=cols_to_drop)

# # 다시 확인 (빈 리스트가 나와야 정상)
# print(
#     "남은 price 관련 컬럼:",
#     [col for col in X_full_clean.columns if 'price' in col.lower()],
# )

제거할 컬럼들: ['category_3_Price_Guides_Publications']
남은 price 관련 컬럼: []


In [ ]:
# cols = pd.Series(X_full_clean.columns)
# for dup in cols[cols.duplicated()].unique():
#     dup_idx = cols[cols == dup].index
#     for i, idx in enumerate(dup_idx):
#         if i != 0:
#             cols[idx] = f"{dup}_{i}"
# X_full_clean.columns = cols

NameError: name 'cols' is not defined

In [3]:
!pip install optuna

In [8]:
# 1. 숫자형(정수, 실수) 데이터 타입인 컬럼만 남기고 나머지는 전부 제외
X_full_clean = X_full_clean.select_dtypes(include=[np.number])

# 2. 혹시 남아있을지 모르는 결측치(NaN)나 무한대 값 처리
X_full_clean = X_full_clean.fillna(0)

print("정리 후 데이터 크기:", X_full_clean.shape)

정리 후 데이터 크기: (1481611, 996)


In [14]:
from scipy.sparse import load_npz
tfidf_all = load_npz("../data/processed/tfidf_all_1481611.npz")

In [15]:
import numpy as np
from scipy.sparse import hstack, csr_matrix

# 정형 피처를 sparse로 변환 후 텍스트 피처와 결합
X_full_clean_sparse = csr_matrix(X_full_clean.values)
X = hstack([X_full_clean_sparse, tfidf_all]).tocsr()

print(X.shape)  # (1481611, 998 + 50000) 형태여야 함

(1481611, 50996)


In [16]:
import optuna
import lightgbm as lgb

def objective(trial):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "num_leaves": trial.suggest_int("num_leaves", 20, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
    }

    model = lgb.train(
        params,
        train_data,
        valid_sets=[valid_data],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    )

    preds_log = model.predict(X_test_f, num_iteration=model.best_iteration)

    # y_log, preds 모두 log1p 스케일이므로 원래 스케일로 복원 후 rmsle 계산
    y_true_orig = np.expm1(y_test_f)
    y_pred_orig = np.expm1(preds_log)

    return rmsle(y_true_orig, y_pred_orig)


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

print("Best RMSLE:", study.best_value)
print("Best params:", study.best_params)

[I 2026-08-27 17:30:15,200] A new study created in memory with name: no-name-638caae9-4adc-4375-8ab8-d6fcf2da5bfa
[I 2026-08-27 17:31:22,208] Trial 0 finished with value: 0.5410923814642387 and parameters: {'num_leaves': 182, 'max_depth': 8, 'learning_rate': 0.10903547658247463, 'feature_fraction': 0.5429859578803498, 'bagging_fraction': 0.5776882461344877, 'bagging_freq': 2, 'min_child_samples': 11, 'lambda_l1': 0.0005600076655686511, 'lambda_l2': 7.995795579498394e-06}. Best is trial 0 with value: 0.5410923814642387.
[I 2026-08-27 17:31:41,626] Trial 1 finished with value: 0.5510494852567904 and parameters: {'num_leaves': 32, 'max_depth': 4, 'learning_rate': 0.09146149111933989, 'feature_fraction': 0.9957481460644428, 'bagging_fraction': 0.5210817662883476, 'bagging_freq': 1, 'min_child_samples': 90, 'lambda_l1': 0.49341910228566743, 'lambda_l2': 3.867931631005112e-07}. Best is trial 0 with value: 0.5410923814642387.
[I 2026-08-27 17:32:03,103] Trial 2 finished with value: 0.54063312

Best RMSLE: 0.5392831351317539
Best params: {'num_leaves': 189, 'max_depth': 6, 'learning_rate': 0.258275658359265, 'feature_fraction': 0.5229304585649578, 'bagging_fraction': 0.9674602351514436, 'bagging_freq': 5, 'min_child_samples': 25, 'lambda_l1': 1.0104952657261795e-05, 'lambda_l2': 0.00563296994118159}
